In [1]:
import cupy as np
# import numpy as np

def rsvd(A, k, p=0, q=0):
    m, n = A.shape
    Omega = np.random.randn(n, k + p)
    Y = A @ Omega
    for _ in range(q):
        Y = A @ (A.T @ Y)
    Q, _ = np.linalg.qr(Y)
    B = Q.T @ A
    U_tilde, S, Vt = np.linalg.svd(B, full_matrices=False)
    U = Q @ U_tilde
    return U[:, :k], S[:k], Vt[:k, :]

def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order = 'F')

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def fold_G(G, Rw, Ru, Rv):
    Gt = np.zeros((Rw, Ru, Rv))
    for r in range(Rv):
        Gt[:, :, r] = G[:, r * Ru : (r + 1) * Ru]
    return Gt

In [2]:
# import numpy as np

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

def forecast(series, mu, W, G, U, V, horizon):
    N, t = series.shape
    d, Rv = V.shape
    Rw = W.shape[1]
    Ru = U.shape[1]
    est = np.zeros((N, horizon))
    for h in range(horizon):
        Xt = np.zeros((N, d))
        for k in range(d):
            Xt[:, k] = series[:, t - horizon + h - (k + 1)]
        est[:, h] = mu + np.einsum('nw, w -> n', W,
                                   np.einsum('wuv, uv -> w',
                                             fold_G(G, Rw, Ru, Rv), U.T @ Xt @ V))
    return est

def forecast(series, mu, W, G, U, V, horizon, step = 1):
    N, t = series.shape
    d, Rv = V.shape
    Rw = W.shape[1]
    Ru = U.shape[1]
    core = fold_G(G, Rw, Ru, Rv)

    def predict_one(buf):
        Xt = buf[:, -1 : -d - 1: -1]
        return mu + W @ np.einsum('wuv, uv -> w', core, U.T @ Xt @ V)

    est = np.zeros((N, horizon))
    for h in range(horizon):
        target = t - horizon + h
        origin = target - step + 1
        buf = series[:, : origin].copy()
        for s in range(step):
            pred = predict_one(buf)
            buf = np.concatenate([buf, pred[:, None]], axis = 1)
        est[:, h] = pred
    return est

In [3]:
def solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu, dv -> uvt', X, U, V)
    C = Y[:, d :] - W @ np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), mat)
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = C - mu[:, np.newaxis]
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################
        mu = np.einsum('nt, nt -> n', theta, C) / np.sum(theta, axis = 1)
    return mu

In [4]:
def ell_W(temp, W):
    return np.einsum('nwk, nw -> nk', temp, W).reshape(-1, order = 'F')

def solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    var0 = np.einsum('nu, ndt, dv -> uvt', U, X, V)
    var1 = np.einsum('wuv, uvt -> wt', fold_G(G, Rw, Ru, Rv), var0)
    temp1 = np.einsum('wt, kt -> wkt', var1, var1)
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, var1)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        w = np.reshape(W, -1, order = 'F')
        response = np.einsum('nt, wt -> nw', theta * data, var1).reshape(-1, order = 'F')
        temp = np.einsum('nt, wkt -> nwk', theta, temp1)
        r = response - ell_W(temp, W)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Rw), order = 'F')
            Aq = ell_W(temp, Q)
            w, r, q, rold = update_cg(w, r, q, Aq, rold)
        W = np.reshape(w, (N, Rw), order = 'F')
    return W

In [6]:
def ell_U(var0, temp1, U):
    return np.einsum('njut, jt -> nu', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('nwut, nu -> wt', var0, U)))

def solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, dv -> nvt', X, V)
    var0 = np.einsum('nvt, wuv -> nwut', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('nwut, nu -> wt', var0, U))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        u = np.reshape(U, -1, order = 'F')
        response = np.einsum('nwut, wt -> nu', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_U(var0, temp1, U), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, Ru), order = 'F')
            Aq = np.reshape(ell_U(var0, temp1, Q), -1, order = 'F')
            u, r, q, rold = update_cg(u, r, q, Aq, rold)
        U = np.reshape(u, (N, Ru), order = 'F')
    return U

In [7]:
def ell_V(var0, temp1, V):
    return np.einsum('djvt, jt -> dv', var0, np.einsum('jwt, wt -> jt', temp1, np.einsum('dwvt, dv -> wt', var0, V)))

def solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N, d, T - d))
    for k in range(d):
        X[:, k, :] = Y[:, d - k - 1 : T - k - 1]
    mat = np.einsum('ndt, nu -> dut', X, U)
    var0 = np.einsum('dut, wuv -> dwvt', mat, fold_G(G, Rw, Ru, Rv))
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - np.einsum('nw, wt -> nt', W, np.einsum('dwvt, dv -> wt', var0, V))
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        temp1 = np.einsum('ij, it, ik -> jkt', W, theta, W)

        ### Conjugate Gradient ###
        v = np.reshape(V, -1, order = 'F')
        response = np.einsum('dwvt, wt -> dv', var0, W.T @ (theta * data))
        r = np.reshape(response - ell_V(var0, temp1, V), -1, order = 'F')
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (d, Rv), order = 'F')
            Aq = np.reshape(ell_V(var0, temp1, Q), -1, order = 'F')
            v, r, q, rold = update_cg(v, r, q, Aq, rold)
        V = np.reshape(v, (d, Rv), order = 'F')
    return V

In [8]:
def left_svd_gram(X, k):
    G = X @ X.T
    evals, evecs = np.linalg.eigh(G)
    idx = np.argsort(evals)[::-1][:k]
    return evecs[:, idx], np.sqrt(np.clip(evals[idx], 0, None))

In [9]:
import cupy as np
import time

def huber_tar(Y, p, delta, d, Rw, Ru, Rv, max_iter = 50):
    N, T = Y.shape
    ##############
    mu = np.zeros(N)
    factor, _, _ = rsvd(Y[:, d :], Rw)
    W = factor[:, : Rw]
    X = np.zeros((N, d * (T - d)))
    for k in range(d):
        X[:, k * (T - d) : (k + 1) * (T - d)] = Y[:, d - k - 1 : T - k - 1]
    factor, _, _ = rsvd(X, Ru)
    U = factor[:, : Ru]
    X = np.zeros((d, N * (T - d)))
    for k in range(d):
        X[k, :] = Y[:, d - k - 1 : T - k - 1].reshape(-1, order = 'F')
    if N * (T - d) > 1e+7:
        V, _ = left_svd_gram(X, Rv)
    else:
        factor, _, _ = rsvd(X, Rv)
        V = factor[:, : Rv]
    tensor_nnb = np.zeros((Rw, Ru, Rv))
    idx = np.arange(Rw)
    tensor_nnb[idx, idx, idx] = 1
    G = ten2mat(tensor_nnb, 0)
    ###############
    for iter in range(max_iter):
        mu = solve_mu(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        W = solve_W(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        U = solve_U(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
        V = solve_V(Y, mu, W, G, U, V, delta, d, Rw, Ru, Rv, p)
    return mu, W, G, U, V

In [46]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

step = 12
for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [3]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : 17 * 24]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon, step)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon, step)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 3
Delta = 40
Running time: 57 seconds
MAE: 244.575
RMSE: 1461.5

MAE: 300.814
RMSE: 1892.46

p = 0.6666666666666666
Rank = 3
Delta = 40
Running time: 56 seconds
MAE: 225.748
RMSE: 1300.1

MAE: 262.307
RMSE: 1758.88

p = 0.5
Rank = 3
Delta = 40
Running time: 56 seconds
MAE: 230.891
RMSE: 1353.09

MAE: 263.405
RMSE: 1793.84

p = 0.3333333333333333
Rank = 3
Delta = 40
Running time: 56 seconds
MAE: 243.927
RMSE: 1295.38

MAE: 287.885
RMSE: 1761.55

p = 0
Rank = 3
Delta = 40
Running time: 56 seconds
MAE: 238.762
RMSE: 1213.49

MAE: 281.344
RMSE: 1676.72



In [47]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

step = 12
for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [4]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : 17 * 24]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, G, U, V, horizon, step)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon, step)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 4
Delta = 40
Running time: 64 seconds
MAE: 898.362
RMSE: 5713.64

MAE: 1690.84
RMSE: 9288.78

p = 0.6666666666666666
Rank = 4
Delta = 40
Running time: 64 seconds
MAE: 897.01
RMSE: 5509.97

MAE: 1633.55
RMSE: 8824.88

p = 0.5
Rank = 4
Delta = 40
Running time: 64 seconds
MAE: 890.612
RMSE: 5390.13

MAE: 1562.71
RMSE: 8421.25

p = 0.3333333333333333
Rank = 4
Delta = 40
Running time: 64 seconds
MAE: 866.307
RMSE: 5144.42

MAE: 1473.79
RMSE: 7889.53

p = 0
Rank = 4
Delta = 40
Running time: 64 seconds
MAE: 835.958
RMSE: 4749.9

MAE: 1386.82
RMSE: 7331.14



In [10]:
import numpy as np

data = np.load('daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [11]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [5]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : - (365 + 365)]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (365 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, G, U, V, horizon)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground != 0)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 5
Delta = 1
Running time: 545 seconds
MAE: 3.375
RMSE: 4.4488

MAE: 3.34328
RMSE: 4.32117

p = 0.6666666666666666
Rank = 5
Delta = 1
Running time: 538 seconds
MAE: 3.18452
RMSE: 4.21574

MAE: 3.22876
RMSE: 4.18847

p = 0.5
Rank = 5
Delta = 1
Running time: 537 seconds
MAE: 3.23308
RMSE: 4.31778

MAE: 3.28136
RMSE: 4.26768

p = 0.3333333333333333
Rank = 5
Delta = 1
Running time: 538 seconds
MAE: 3.35876
RMSE: 4.47137

MAE: 3.34454
RMSE: 4.36574

p = 0
Rank = 5
Delta = 1
Running time: 537 seconds
MAE: 2.95785
RMSE: 3.9387

MAE: 2.96311
RMSE: 3.91183



In [12]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [6]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, G, U, V = huber_tar(np.array(data[:, : - (365 + 365)]), p, delta, d, R, R, R, 50)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (365 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, G, U, V, horizon)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, G, U, V, horizon)
    pos = np.where(ground != 0)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 6
Delta = 1
Running time: 775 seconds
MAE: 3.46567
RMSE: 4.58117

MAE: 3.49441
RMSE: 4.4981

p = 0.6666666666666666
Rank = 6
Delta = 1
Running time: 774 seconds
MAE: 3.23325
RMSE: 4.29891

MAE: 3.32083
RMSE: 4.31147

p = 0.5
Rank = 6
Delta = 1
Running time: 774 seconds
MAE: 3.30565
RMSE: 4.4384

MAE: 3.32972
RMSE: 4.3608

p = 0.3333333333333333
Rank = 6
Delta = 1
Running time: 774 seconds
MAE: 3.26432
RMSE: 4.38621

MAE: 3.31753
RMSE: 4.35019

p = 0
Rank = 6
Delta = 1
Running time: 774 seconds
MAE: 2.75034
RMSE: 3.74132

MAE: 2.771
RMSE: 3.7035

